In [1]:
## Dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.interpolate import interp1d
import random
from shapely.geometry import Point
from shapely.geometry import mapping
import pickle
import xarray as xr
import rioxarray
from pandas.tseries.offsets import DateOffset
import geopandas as gpd

pd.set_option('display.max_columns', None)

## Set working directory to tree_growth_metaanalysis_snmc
os.chdir('../../')


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/global/software/rocky-8.x86_64/manual/modules/langs/anaconda3/2024.02-1/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/global/software/rocky-8.x86_64/manual/modules/langs/anaconda3/2024.02-1/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/global/software/rocky-8.x86_64/manual/mo

AttributeError: _ARRAY_API not found

In [2]:
# Import utils
from Inventory_data_processing.Code.config import *
from covariate_functions import *

In [3]:
def cvar_mean_by_plot(cvar, df):
    '''Loops through sites; for each site, finds the mean value of the climate variable for each 
    measurement unit (plot) over the study timeframe'''
    site_list = []
    plot_list = []
    val_list = []
    unit_nms = []
    
    for site in df['Site'].unique():
        # Subset by initial and final year of the study
        yri = start_years[site] + 1 # Include the next climate year but not the previous
        yrf = end_years[site]
        data_subset = df.loc[(df['Site']==site) & (df['year']>=yri) & (df['year']<=yrf)]

        ## Loop through units
        for u in data_subset['UnitID'].unique():
            ## Loop through subplots
          for p in data_subset.loc[data_subset['UnitID']==u, 'PlotID'].unique():
              site_list = site_list + [site]
              unit_nms = unit_nms + [u]
              plot_list = plot_list + data_subset.loc[(data_subset['UnitID']==u) & (data_subset['PlotID']==p), 'unique_nm'].unique().tolist()
              # Mean value of climate variable
              val_list = val_list + [data_subset.loc[(data_subset['UnitID']==u) & (data_subset['PlotID']==p), cvar].mean()]
        
    mean_plot_cvar = pd.DataFrame({'Site':site_list, 'UnitID':unit_nms, 'unique_nm':plot_list, cvar:val_list})
    return mean_plot_cvar

# Load site data

In [4]:
## Get lat-lon coordinates of each experimental unit
plots = pd.read_csv('Covariate_and_metaregressor_processing/Input_data/unit_lat_lon.csv')

## Assign each plot a unique name
plots['unique_nm'] = plots[['Site', 'UnitID', 'PlotID']].T.agg('_'.join)

plots

,PlotID,Latitude,Longitude,Site,UnitID,unique_nm
0,FFS7CONTROL,36.581269,-118.762843,Sequoia,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL
1,FFS5BURN,36.584807,-118.755245,Sequoia,FFS5BURN,Sequoia_FFS5BURN_FFS5BURN
2,FFS6BURN,36.585185,-118.759557,Sequoia,FFS6BURN,Sequoia_FFS6BURN_FFS6BURN
3,FFS2BURN,36.595060,-118.746970,Sequoia,FFS2BURN,Sequoia_FFS2BURN_FFS2BURN
4,LOTHAR,36.561899,-118.742603,TharpsCreek,LOTHAR,TharpsCreek_LOTHAR_LOTHAR
...,...,...,...,...,...,...
556,217,40.629233,-121.668217,LaTour,217,LaTour_217_217
557,218,40.625433,-121.668600,LaTour,218,LaTour_218_218
558,219,40.621633,-121.668300,LaTour,219,LaTour_219_219
559,220,40.618417,-121.668467,LaTour,220,LaTour_220_220


In [5]:
# Specify time periods of interest
start_yr = 1980
end_yr = 2024

winter_months = [10, 11, 12, 1, 2] # a climate year runs Oct-Sept
summer_months = [6, 7, 8, 9]

# Winter precipitation meta-regressor

In [6]:
# Load precipitation by climate year
ds = xr.open_dataset('Covariate_and_metaregressor_processing/Processed_data/Climate/ppt_by_waterYr.nc', decode_times=False)
ds = ds.rename({'latitude':'lat', 'longitude':'lon'})
ds

<xarray.Dataset> Size: 10MB
Dimensions:         (lon: 248, lat: 232, time: 45)
Coordinates:
  * lon             (lon) float64 2kB -124.4 -124.4 -124.4 ... -114.2 -114.1
  * lat             (lat) float64 2kB 42.06 42.02 41.98 ... 32.52 32.48 32.44
  * time            (time) float64 360B 10.0 11.0 12.0 13.0 ... 52.0 53.0 54.0
Data variables:
    ppt_by_waterYr  (time, lat, lon) float32 10MB ...
    crs             int32 4B ...
Attributes:
    Conventions:  CF-1.4
    created_by:   R packages ncdf4 and terra (version 1.7-71)
    date:         2026-01-01 13:38:44

In [7]:
# Find climate grid cell associated with each plot
ppt_master = find_plot_gridcells(ds, plots)
ppt_master['year'] = ppt_master['time'] + 1970
ppt_master

,lon,lat,time,ppt_by_waterYr,crs,Site,UnitID,PlotID,unique_nm,year
0,-118.770833,36.562500,10.0,550.799988,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1980.0
1,-118.770833,36.562500,11.0,303.899994,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1981.0
2,-118.770833,36.562500,12.0,434.700012,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1982.0
3,-118.770833,36.562500,13.0,1012.900024,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1983.0
4,-118.770833,36.562500,14.0,512.200012,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1984.0
...,...,...,...,...,...,...,...,...,...,...
25240,-121.645833,40.604167,50.0,349.700012,-2147483647,LaTour,221,221,LaTour_221_221,2020.0
25241,-121.645833,40.604167,51.0,380.000000,-2147483647,LaTour,221,221,LaTour_221_221,2021.0
25242,-121.645833,40.604167,52.0,652.799988,-2147483647,LaTour,221,221,LaTour_221_221,2022.0
25243,-121.645833,40.604167,53.0,774.900024,-2147483647,LaTour,221,221,LaTour_221_221,2023.0


In [8]:
# Get mean variable value across years of study for each unit
mean_plot_ppt = cvar_mean_by_plot('ppt_by_waterYr', ppt_master)
mean_plot_ppt

,Site,UnitID,unique_nm,ppt_by_waterYr
0,Sequoia,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,441.946686
1,Sequoia,FFS5BURN,Sequoia_FFS5BURN_FFS5BURN,463.420013
2,Sequoia,FFS6BURN,Sequoia_FFS6BURN_FFS6BURN,463.420013
3,Sequoia,FFS2BURN,Sequoia_FFS2BURN_FFS2BURN,443.133362
4,TharpsCreek,LOTHAR,TharpsCreek_LOTHAR_LOTHAR,508.322540
...,...,...,...,...
556,LaTour,217,LaTour_217_217,665.066711
557,LaTour,218,LaTour_218_218,665.066711
558,LaTour,219,LaTour_219_219,689.833374
559,LaTour,220,LaTour_220_220,689.833374


In [9]:
# ## Subset units that are included in analysis
### I don't do this because we didn't do this step for processing the CWD data for SSMs

# ppt_unit_subset = pd.DataFrame()

# for site in units:
#     temp = mean_plot_ppt.loc[(mean_plot_ppt['UnitID'].isin([str(i) for i in units[site]])) & (mean_plot_ppt['Site']==site)]
#     ppt_unit_subset = pd.concat([ppt_unit_subset, temp])
# ppt_unit_subset

In [10]:
## Plot-level to unit-level
mean_unit_pr = mean_plot_ppt.drop(columns='unique_nm').groupby(['Site', 'UnitID'], as_index=False).agg('mean')
mean_unit_pr

,Site,UnitID,ppt_by_waterYr
0,Blodgett,180,990.653015
1,Blodgett,190,957.423523
2,Blodgett,240,982.345581
3,Blodgett,340,990.653015
4,Blodgett,350,990.653015
...,...,...,...
284,WLakeTahoe,TWC 3 T,544.255493
285,WLakeTahoe,WRD 20-16 C,592.488892
286,WLakeTahoe,WRD 20-16 T,592.488892
287,WLakeTahoe,WRD 20-9 C,592.488892


In [11]:
## Unit-level to site-level
mean_site_pr = mean_unit_pr.drop(columns='UnitID').groupby('Site', as_index=False).agg('mean')
mean_site_pr

,Site,ppt_by_waterYr
0,Blodgett,959.960510
1,LaTour,695.896973
2,STEF,599.031311
3,Sequoia,452.980011
4,Teakettle,570.529419
5,TharpsCreek,508.322540
6,WLakeTahoe,571.577759


# Summer maximum temperature meta-regressor

In [12]:
# Load tmax by climate year
tmax_by_yr = xr.open_dataset('Covariate_and_metaregressor_processing/Processed_data/Climate/tmax_by_waterYr.nc', 
                             decode_times=False)
tmax_by_yr = tmax_by_yr.rename({'latitude':'lat', 'longitude':'lon'})
tmax_by_yr

<xarray.Dataset> Size: 10MB
Dimensions:          (lon: 248, lat: 232, time: 44)
Coordinates:
  * lon              (lon) float64 2kB -124.4 -124.4 -124.4 ... -114.2 -114.1
  * lat              (lat) float64 2kB 42.06 42.02 41.98 ... 32.52 32.48 32.44
  * time             (time) float64 352B 10.0 11.0 12.0 13.0 ... 51.0 52.0 53.0
Data variables:
    tmax_by_waterYr  (time, lat, lon) float32 10MB ...
    crs              int32 4B ...
Attributes:
    Conventions:  CF-1.4
    created_by:   R packages ncdf4 and terra (version 1.7-71)
    date:         2026-01-01 13:38:21

In [13]:
# Find climate grid cell associated with each plot
tmax_master = find_plot_gridcells(tmax_by_yr, plots)
tmax_master['year'] = tmax_master['time'] + 1970
tmax_master

,lon,lat,time,tmax_by_waterYr,crs,Site,UnitID,PlotID,unique_nm,year
0,-118.770833,36.562500,10.0,23.197500,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1980.0
1,-118.770833,36.562500,11.0,25.607500,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1981.0
2,-118.770833,36.562500,12.0,22.212500,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1982.0
3,-118.770833,36.562500,13.0,22.957500,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1983.0
4,-118.770833,36.562500,14.0,24.512501,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1984.0
...,...,...,...,...,...,...,...,...,...,...
24679,-121.645833,40.604167,49.0,23.475000,-2147483647,LaTour,221,221,LaTour_221_221,2019.0
24680,-121.645833,40.604167,50.0,24.549999,-2147483647,LaTour,221,221,LaTour_221_221,2020.0
24681,-121.645833,40.604167,51.0,25.549999,-2147483647,LaTour,221,221,LaTour_221_221,2021.0
24682,-121.645833,40.604167,52.0,25.049999,-2147483647,LaTour,221,221,LaTour_221_221,2022.0


In [14]:
# Get mean variable value across years of study for each unit
mean_plot_tmax = cvar_mean_by_plot('tmax_by_waterYr', tmax_master)
mean_plot_tmax

,Site,UnitID,unique_nm,tmax_by_waterYr
0,Sequoia,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,24.187670
1,Sequoia,FFS5BURN,Sequoia_FFS5BURN_FFS5BURN,21.608665
2,Sequoia,FFS6BURN,Sequoia_FFS6BURN_FFS6BURN,21.608665
3,Sequoia,FFS2BURN,Sequoia_FFS2BURN_FFS2BURN,20.254002
4,TharpsCreek,LOTHAR,TharpsCreek_LOTHAR_LOTHAR,22.714918
...,...,...,...,...
556,LaTour,217,LaTour_217_217,23.387667
557,LaTour,218,LaTour_218_218,23.387667
558,LaTour,219,LaTour_219_219,24.332335
559,LaTour,220,LaTour_220_220,24.332335


In [15]:
# ## Optional: Subset units that are included in analysis
# tmax_unit_subset = pd.DataFrame()

# for site in units:
#     temp = mean_plot_tmax.loc[(mean_plot_tmax['UnitID'].isin([str(i) for i in units[site]])) & (mean_plot_tmax['Site']==site)]
#     tmax_unit_subset = pd.concat([tmax_unit_subset, temp])
# tmax_unit_subset

In [16]:
## Plot-level to unit-level
mean_unit_tmax = mean_plot_tmax.drop(columns='unique_nm').groupby(['Site', 'UnitID'], as_index=False).agg('mean')
mean_unit_tmax

,Site,UnitID,tmax_by_waterYr
0,Blodgett,180,27.263235
1,Blodgett,190,28.117352
2,Blodgett,240,27.476765
3,Blodgett,340,27.263235
4,Blodgett,350,27.263235
...,...,...,...
284,WLakeTahoe,TWC 3 T,23.616549
285,WLakeTahoe,WRD 20-16 C,22.754723
286,WLakeTahoe,WRD 20-16 T,22.754723
287,WLakeTahoe,WRD 20-9 C,22.754723


In [17]:
## Unit-level to site-level
mean_site_tmax = mean_unit_tmax.drop(columns='UnitID').groupby('Site', as_index=False).agg('mean')
mean_site_tmax

,Site,tmax_by_waterYr
0,Blodgett,27.728548
1,LaTour,24.285917
2,STEF,25.354879
3,Sequoia,21.914751
4,Teakettle,23.081861
5,TharpsCreek,22.714918
6,WLakeTahoe,23.189638


# CWD meta-regressor

In [18]:
# Load cwd by climate year
cwd_by_yr = xr.open_dataset('Covariate_and_metaregressor_processing/Processed_data/Climate/cwd_by_waterYr.nc', 
                            decode_times=False)
cwd_by_yr = cwd_by_yr.rename({'latitude':'lat', 'longitude':'lon'})
cwd_by_yr

<xarray.Dataset> Size: 10MB
Dimensions:         (lon: 248, lat: 232, time: 45)
Coordinates:
  * lon             (lon) float64 2kB -124.4 -124.4 -124.4 ... -114.2 -114.1
  * lat             (lat) float64 2kB 42.06 42.02 41.98 ... 32.52 32.48 32.44
  * time            (time) float64 360B 10.0 11.0 12.0 13.0 ... 52.0 53.0 54.0
Data variables:
    cwd_by_waterYr  (time, lat, lon) float32 10MB ...
    crs             int32 4B ...
Attributes:
    Conventions:  CF-1.4
    created_by:   R packages ncdf4 and terra (version 1.7-71)
    date:         2026-01-01 13:51:54

In [19]:
# Find climate grid cell associated with each plot
cwd_master = find_plot_gridcells(cwd_by_yr, plots)
cwd_master['year'] = cwd_master['time'] + 1970
cwd_master

,lon,lat,time,cwd_by_waterYr,crs,Site,UnitID,PlotID,unique_nm,year
0,-118.770833,36.562500,10.0,577.000000,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1980.0
1,-118.770833,36.562500,11.0,824.900024,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1981.0
2,-118.770833,36.562500,12.0,518.599976,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1982.0
3,-118.770833,36.562500,13.0,477.899994,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1983.0
4,-118.770833,36.562500,14.0,824.400024,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1984.0
...,...,...,...,...,...,...,...,...,...,...
25240,-121.645833,40.604167,50.0,578.200012,-2147483647,LaTour,221,221,LaTour_221_221,2020.0
25241,-121.645833,40.604167,51.0,690.299988,-2147483647,LaTour,221,221,LaTour_221_221,2021.0
25242,-121.645833,40.604167,52.0,505.500000,-2147483647,LaTour,221,221,LaTour_221_221,2022.0
25243,-121.645833,40.604167,53.0,406.399994,-2147483647,LaTour,221,221,LaTour_221_221,2023.0


In [20]:
# Get mean variable value across years of study for each unit
mean_plot_cwd = cvar_mean_by_plot('cwd_by_waterYr', cwd_master)
mean_plot_cwd

,Site,UnitID,unique_nm,cwd_by_waterYr
0,Sequoia,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,859.166626
1,Sequoia,FFS5BURN,Sequoia_FFS5BURN_FFS5BURN,664.253296
2,Sequoia,FFS6BURN,Sequoia_FFS6BURN_FFS6BURN,664.253296
3,Sequoia,FFS2BURN,Sequoia_FFS2BURN_FFS2BURN,607.720032
4,TharpsCreek,LOTHAR,TharpsCreek_LOTHAR_LOTHAR,743.248352
...,...,...,...,...
556,LaTour,217,LaTour_217_217,467.966705
557,LaTour,218,LaTour_218_218,467.966705
558,LaTour,219,LaTour_219_219,492.753357
559,LaTour,220,LaTour_220_220,492.753357


In [21]:
# ## Optional: Subset units that are included in analysis
# cwd_unit_subset = pd.DataFrame()

# for site in units:
#     temp = mean_plot_cwd.loc[(mean_plot_cwd['UnitID'].isin([str(i) for i in units[site]])) & (mean_plot_cwd['Site']==site)]
#     cwd_unit_subset = pd.concat([cwd_unit_subset, temp])
# cwd_unit_subset

In [22]:
## Plot-level to unit-level
mean_unit_cwd = mean_plot_cwd.drop(columns='unique_nm').groupby(['Site', 'UnitID'], as_index=False).agg('mean')
mean_unit_cwd

,Site,UnitID,cwd_by_waterYr
0,Blodgett,180,526.964661
1,Blodgett,190,545.017639
2,Blodgett,240,531.477905
3,Blodgett,340,526.964661
4,Blodgett,350,526.964661
...,...,...,...
284,WLakeTahoe,TWC 3 T,627.355530
285,WLakeTahoe,WRD 20-16 C,568.055542
286,WLakeTahoe,WRD 20-16 T,568.055542
287,WLakeTahoe,WRD 20-9 C,568.055542


In [23]:
## Unit-level to site-level
mean_site_cwd = mean_unit_cwd.drop(columns='UnitID').groupby('Site', as_index=False).agg('mean')
mean_site_cwd

,Site,cwd_by_waterYr
0,Blodgett,542.300293
1,LaTour,484.532257
2,STEF,705.645203
3,Sequoia,698.848328
4,Teakettle,742.972534
5,TharpsCreek,743.248352
6,WLakeTahoe,594.393066


# PDSI meta-regressor

In [24]:
# Load PDSI by climate year
pdsi_by_yr = xr.open_dataset('Covariate_and_metaregressor_processing/Processed_data/Climate/pdsi_by_waterYr.nc', 
                             decode_times=False)
pdsi_by_yr = pdsi_by_yr.rename({'latitude':'lat', 'longitude':'lon'})
pdsi_by_yr

<xarray.Dataset> Size: 10MB
Dimensions:          (lon: 248, lat: 232, time: 45)
Coordinates:
  * lon              (lon) float64 2kB -124.4 -124.4 -124.4 ... -114.2 -114.1
  * lat              (lat) float64 2kB 42.06 42.02 41.98 ... 32.52 32.48 32.44
  * time             (time) float64 360B 10.0 11.0 12.0 13.0 ... 52.0 53.0 54.0
Data variables:
    pdsi_by_waterYr  (time, lat, lon) float32 10MB ...
    crs              int32 4B ...
Attributes:
    Conventions:  CF-1.4
    created_by:   R packages ncdf4 and terra (version 1.7-71)
    date:         2026-01-01 13:39:08

In [25]:
# Find climate grid cell associated with each plot
pdsi_master = find_plot_gridcells(pdsi_by_yr, plots)
pdsi_master['year'] = pdsi_master['time'] + 1970
pdsi_master

,lon,lat,time,pdsi_by_waterYr,crs,Site,UnitID,PlotID,unique_nm,year
0,-118.770833,36.562500,10.0,1.561111,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1980.0
1,-118.770833,36.562500,11.0,-0.982500,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1981.0
2,-118.770833,36.562500,12.0,1.605000,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1982.0
3,-118.770833,36.562500,13.0,6.947500,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1983.0
4,-118.770833,36.562500,14.0,2.143333,-2147483647,Sequoia,FFS7CONTROL,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,1984.0
...,...,...,...,...,...,...,...,...,...,...
25240,-121.645833,40.604167,50.0,-1.501667,-2147483647,LaTour,221,221,LaTour_221_221,2020.0
25241,-121.645833,40.604167,51.0,-4.110833,-2147483647,LaTour,221,221,LaTour_221_221,2021.0
25242,-121.645833,40.604167,52.0,-2.825000,-2147483647,LaTour,221,221,LaTour_221_221,2022.0
25243,-121.645833,40.604167,53.0,0.346667,-2147483647,LaTour,221,221,LaTour_221_221,2023.0


In [26]:
# Get mean variable value across years of study for each unit
mean_plot_pdsi = cvar_mean_by_plot('pdsi_by_waterYr', pdsi_master)
mean_plot_pdsi

,Site,UnitID,unique_nm,pdsi_by_waterYr
0,Sequoia,FFS7CONTROL,Sequoia_FFS7CONTROL_FFS7CONTROL,-2.394389
1,Sequoia,FFS5BURN,Sequoia_FFS5BURN_FFS5BURN,-2.084722
2,Sequoia,FFS6BURN,Sequoia_FFS6BURN_FFS6BURN,-2.084722
3,Sequoia,FFS2BURN,Sequoia_FFS2BURN_FFS2BURN,-2.020444
4,TharpsCreek,LOTHAR,TharpsCreek_LOTHAR_LOTHAR,-0.911075
...,...,...,...,...
556,LaTour,217,LaTour_217_217,-0.746167
557,LaTour,218,LaTour_218_218,-0.746167
558,LaTour,219,LaTour_219_219,-0.670556
559,LaTour,220,LaTour_220_220,-0.670556


In [27]:
# ## Optional: Subset units that are included in analysis
# pdsi_unit_subset = pd.DataFrame()

# for site in units:
#     temp = mean_plot_pdsi.loc[(mean_plot_pdsi['UnitID'].isin([str(i) for i in units[site]])) & (mean_plot_pdsi['Site']==site)]
#     pdsi_unit_subset = pd.concat([pdsi_unit_subset, temp])
# pdsi_unit_subset

In [28]:
## Plot-level to unit-level
mean_unit_pdsi = mean_plot_pdsi.drop(columns='unique_nm').groupby(['Site', 'UnitID'], as_index=False).agg('mean')
mean_unit_pdsi

,Site,UnitID,pdsi_by_waterYr
0,Blodgett,180,-0.323137
1,Blodgett,190,-0.313284
2,Blodgett,240,-0.320674
3,Blodgett,340,-0.323137
4,Blodgett,350,-0.323137
...,...,...,...
284,WLakeTahoe,TWC 3 T,-1.146706
285,WLakeTahoe,WRD 20-16 C,-1.129444
286,WLakeTahoe,WRD 20-16 T,-1.129444
287,WLakeTahoe,WRD 20-9 C,-1.129444


In [29]:
## Unit-level to site-level
mean_site_pdsi = mean_unit_pdsi.drop(columns='UnitID').groupby('Site', as_index=False).agg('mean')
mean_site_pdsi

,Site,pdsi_by_waterYr
0,Blodgett,-0.319073
1,LaTour,-0.671427
2,STEF,-1.640567
3,Sequoia,-2.146069
4,Teakettle,-1.041993
5,TharpsCreek,-0.911075
6,WLakeTahoe,-1.135433


# Merge data

In [30]:
merged_cvars = mean_site_pr[['Site']]
for i in [mean_site_pr, mean_site_tmax, mean_site_cwd, mean_site_pdsi]:
  merged_cvars = merged_cvars.merge(i, on = 'Site')

merged_cvars

,Site,ppt_by_waterYr,tmax_by_waterYr,cwd_by_waterYr,pdsi_by_waterYr
0,Blodgett,959.960510,27.728548,542.300293,-0.319073
1,LaTour,695.896973,24.285917,484.532257,-0.671427
2,STEF,599.031311,25.354879,705.645203,-1.640567
3,Sequoia,452.980011,21.914751,698.848328,-2.146069
4,Teakettle,570.529419,23.081861,742.972534,-1.041993
5,TharpsCreek,508.322540,22.714918,743.248352,-0.911075
6,WLakeTahoe,571.577759,23.189638,594.393066,-1.135433


In [31]:
# Save to disk
merged_cvars.to_csv('Covariate_and_metaregressor_processing/Processed_data/Climate/site_mean_clim.csv', 
                    index=False)